# Atlas Weather Intelligence Platform — Data Science Notebook

**PM Accelerator AI Engineer Internship Assessment — Dual Role (Backend Engineer + Data Science)**

**Candidate:** Maryam Shanabli
**Submission date:** 18-6-2026
**Repository:** Atlas — FastAPI + PostgreSQL weather platform with a trained forecasting model and statistical anomaly detection, exposed through 14+ REST endpoints documented at `/docs`.

## About PM Accelerator

PM Accelerator is the world's most accessible product management program, providing aspiring and experienced PMs with the skills, mentorship, and real-world experience needed to land and excel in product management roles. The program pairs candidates with real projects — like this one — that mirror the cross-functional, dual-role work (technical build + data-driven decision making) PMs are expected to lead in industry.

This notebook is the data-science deliverable for the assessment. It documents, end-to-end and reproducibly, the exact pipeline that produced the three model artifacts the live API loads at startup (`atlas/models/forecast_model.joblib`, `location_baselines.joblib`, `known_coverage_coords.joblib`). The pipeline logic itself lives in `atlas/notebooks/data_cleaning.py`, `anomaly_detection.py`, and `forecasting.py` as importable, unit-testable modules — this notebook imports and runs those modules rather than duplicating their logic, so there is exactly one source of truth for the data science code.

## Contents

1. Setup & data loading
2. Exploratory data analysis
3. Data cleaning — two distinct problems, two distinct fixes
4. Anomaly detection — per-location statistical baselines
5. Forecasting model — seasonal-naive baseline vs. gradient boosting
6. Saving model artifacts (what the FastAPI app actually loads)
7. Conclusions & architecture recap


## 1. Setup & Data Loading

**Dataset:** [Global Weather Repository](https://www.kaggle.com/datasets/nelgiriyewithana/global-weather-repository) (Kaggle) — daily-updated current-conditions snapshot for ~250 cities worldwide, including temperature, air quality, and astronomical fields.

If you don't already have the CSV locally, download it from the link above and place it at `notebooks/data/GlobalWeatherRepository.csv`. This is the exact file this pipeline was developed and verified against (see handoff notes: data_cleaning.py, anomaly_detection.py and forecasting.py were each run end-to-end on this real CSV before this notebook was written).

In [ ]:
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Make the verified pipeline modules importable. This notebook lives in
# atlas/notebooks/, and the modules it imports (data_cleaning.py,
# anomaly_detection.py, forecasting.py) live right next to it.
NOTEBOOK_DIR = Path.cwd() if (Path.cwd() / "data_cleaning.py").exists() else Path.cwd() / "notebooks"
sys.path.insert(0, str(NOTEBOOK_DIR))

from data_cleaning import clean_dataset, PHYSICAL_LIMITS
from anomaly_detection import build_location_baselines, check_anomaly
from forecasting import (
    add_time_features, seasonal_naive_predict, train_gradient_boosting, evaluate,
)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

DATA_PATH = NOTEBOOK_DIR / "data" / "GlobalWeatherRepository.csv"
MODELS_DIR = NOTEBOOK_DIR.parent / "models"

df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df):,} rows, {df.shape[1]} columns")
df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include="number").T

## 2. Exploratory Data Analysis

Before cleaning anything, look at the raw data closely enough to find the actual problems in it — not generic ones. Two issues surfaced during inspection and drive everything in Section 3:

1. **The same country appears under multiple spellings/languages** at what is clearly the same physical location (different rows, identical or near-identical lat/long).
2. **A handful of readings are physically impossible** (e.g. temperatures or pressures outside any value ever recorded on Earth), which look like sensor or parsing errors rather than real weather.

In [ ]:
# Country-name contamination: how many distinct country labels do we see,
# vs how many distinct physical locations (rounded lat/long)?
n_country_labels = df["country"].nunique()
n_locations = df[["latitude", "longitude"]].round(2).drop_duplicates().shape[0]
print(f"Distinct country label strings: {n_country_labels}")
print(f"Distinct physical locations (lat/long rounded to 2dp): {n_locations}")
print(f"Distinct location_name values: {df['location_name'].nunique()}")

In [ ]:
# A quick look at non-ASCII / non-English country labels — these are the
# ones canonicalize_countries() in data_cleaning.py resolves via coordinate
# clustering (see Section 3).
import re
non_ascii_countries = df.loc[~df["country"].str.fullmatch(r"[A-Za-z\s\.\-']+"), "country"].unique()
print(f"{len(non_ascii_countries)} non-English-looking country labels found, e.g.:")
print(sorted(non_ascii_countries)[:15])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df["temperature_celsius"], bins=50, ax=axes[0])
axes[0].set_title("Temperature (°C) — raw distribution")
sns.histplot(df["pressure_mb"], bins=50, ax=axes[1])
axes[1].set_title("Pressure (mb) — raw distribution")
plt.tight_layout()
plt.show()

In [ ]:
# Physical-limit violations, by field, using the same limits the
# pipeline itself uses (data_cleaning.PHYSICAL_LIMITS) so this EDA and the
# actual cleaning step in Section 3 can never silently drift apart.
for col, (low, high) in PHYSICAL_LIMITS.items():
    n_bad = (~df[col].between(low, high)).sum()
    print(f"{col:25s} valid range [{low}, {high}]  ->  {n_bad} rows outside it")

In [ ]:
print("Missing values per column (top 10):")
df.isna().sum().sort_values(ascending=False).head(10)

## 3. Data Cleaning — Two Problems, Two Fixes

The cleaning logic itself lives in `data_cleaning.py` and is intentionally **not** generic boilerplate (drop-nulls-and-z-score-everything). It encodes two specific, deliberately separate fixes:

**Country name contamination → coordinate clustering.** Rows within ~1km of each other (lat/long rounded to 2 decimals) are treated as the same physical place, and the cleanest-looking label among them — preferring an ASCII, plain-English name — is chosen as the canonical `country_clean`. A small manual lookup table closes the few singleton cases clustering alone can't resolve (e.g. a label that exists only in one language anywhere in the dataset).

**Physically impossible readings → flagged, not dropped.** Temperature, wind speed, pressure, and humidity are checked against known physical limits for Earth's surface (e.g. no recorded surface temperature has ever exceeded ~56.7°C). This is deliberately *not* a statistical/z-score approach, because a z-score would also flag genuinely extreme-but-real readings (e.g. very high PM2.5 during a wildfire smoke event) — and conflating "physically impossible" with "unusual" would throw away real signal that Section 4's anomaly detection is specifically designed to surface. Flagged rows are kept and marked, not silently dropped, so a reviewer can see exactly what was caught and why.

In [ ]:
df_clean = clean_dataset(df)

print(f"Country labels before cleaning: {df['country'].nunique()}")
print(f"Country labels after cleaning (country_clean): {df_clean['country_clean'].nunique()}")
print(f"Physically implausible rows flagged: {df_clean['any_implausible_reading'].sum()} "
      f"of {len(df_clean)} ({df_clean['any_implausible_reading'].mean():.2%})")
df_clean[["location_name", "country", "country_clean", "any_implausible_reading"]].head()

In [ ]:
# Spot-check: locations where the raw country label and the cleaned label differ
mismatches = df_clean[df_clean["country"] != df_clean["country_clean"]]
mismatches[["location_name", "country", "country_clean"]].drop_duplicates().head(15)

## 4. Anomaly Detection — Per-Location Statistical Baselines

This is deliberately a **different** notion of "anomalous" than Section 3's physical-limit flags:

- *Physically implausible* = always an error (a reading that violates a hard physical limit on Earth).
- *Statistically unusual* = a reading that is far from **this specific location's own** historical distribution, while remaining entirely physically plausible — e.g. an unusually hot day for Reykjavik, or elevated PM2.5 during a regional smoke event. This is real signal, and it's what the live `/anomaly-check/{location_id}` endpoint reports to API users.

Baselines are built only from physically plausible rows (Section 3's flag), then a live reading is compared against its location's mean/std using a z-score. `z_threshold=2.5` is a deliberate choice — at roughly the 99th percentile under a normal approximation, it flags genuinely extreme readings without firing on routine day-to-day variation.

In [ ]:
baselines = build_location_baselines(df_clean)
print(f"Computed baselines for {len(baselines)} (location, country) pairs")
baselines.sort_values("n_observations", ascending=False).head(10)

In [ ]:
# Demo: check_anomaly() against a real baseline row, for both a normal
# reading and a deliberately extreme one.
sample_row = baselines.iloc[0]
print(f"Baseline for {sample_row['location_name']}, {sample_row['country_clean']}: "
      f"mean={sample_row['temp_mean']:.1f}°C, std={sample_row['temp_std']:.1f}")

normal_reading = {"temperature_celsius": sample_row["temp_mean"], "air_quality_PM2.5": sample_row["pm25_mean"]}
extreme_reading = {"temperature_celsius": sample_row["temp_mean"] + 6 * (sample_row["temp_std"] or 1), "air_quality_PM2.5": sample_row["pm25_mean"]}

print("Normal reading verdict: ", check_anomaly(normal_reading, sample_row))
print("Extreme reading verdict:", check_anomaly(extreme_reading, sample_row))

## 5. Forecasting Model — Seasonal-Naive Baseline vs. Gradient Boosting

Two models are trained and compared, deliberately — a real model only earns its place in production if it beats a sensible baseline:

1. **Seasonal-naive baseline.** Predicts a day's temperature as the historical mean temperature for that (location, day-of-year) combination, falling back to the location's overall mean if that exact day-of-year wasn't observed. No machine learning — just "what does this place usually look like on this day."
2. **Gradient boosting regressor.** Trained on `latitude`, `longitude`, and day-of-year encoded as `sin`/`cos` (to capture seasonality without a discontinuity at the year boundary). Critically, because it uses lat/long as features rather than a per-location lookup table, it can produce a forecast for **any** coordinates — including locations never seen in training — which is exactly what `has_model_coverage` on the `locations` table distinguishes (interpolated estimate vs. a location with real historical baselines).

In [ ]:
# Time-based train/test split: train on the first 80% of dates, test on the
# most recent 20%. This is the correct split for a forecasting model — a
# random row split would leak future information into training via rows
# from the same location on nearby dates.
df_model = df_clean[~df_clean["any_implausible_reading"]].copy()
df_model["last_updated"] = pd.to_datetime(df_model["last_updated"])
df_model = df_model.sort_values("last_updated")

split_idx = int(len(df_model) * 0.8)
split_date = df_model.iloc[split_idx]["last_updated"]
train_df = df_model[df_model["last_updated"] < split_date]
test_df = df_model[df_model["last_updated"] >= split_date]
print(f"Train: {len(train_df):,} rows (through {train_df['last_updated'].max().date()})")
print(f"Test:  {len(test_df):,} rows (from {test_df['last_updated'].min().date()})")

In [ ]:
naive_preds = seasonal_naive_predict(train_df, test_df)
naive_metrics = evaluate(test_df["temperature_celsius"].values, naive_preds)
print("Seasonal-naive baseline:", naive_metrics)

In [ ]:
model, features = train_gradient_boosting(train_df)

test_features = add_time_features(test_df)[features]
gb_preds = model.predict(test_features)
gb_metrics = evaluate(test_df["temperature_celsius"].values, gb_preds)
print("Gradient boosting model:", gb_metrics)

improvement = (naive_metrics["mae"] - gb_metrics["mae"]) / naive_metrics["mae"] * 100
print(f"\nMAE improvement over seasonal-naive baseline: {improvement:.1f}%")

In [ ]:
comparison = pd.DataFrame([
    {"model": "Seasonal-naive baseline", **naive_metrics},
    {"model": "Gradient boosting", **gb_metrics},
]).set_index("model")
comparison.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
comparison[["mae", "rmse"]].plot(kind="bar", ax=ax)
ax.set_ylabel("°C")
ax.set_title("Forecast error: seasonal-naive vs. gradient boosting")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance — confirms the model is leaning on seasonality
# (doy_sin/doy_cos) and geography (latitude) as expected, not noise.
importance = pd.Series(model.feature_importances_, index=features).sort_values(ascending=False)
importance

## 6. Saving Model Artifacts

These three files are exactly what `atlas/app/ml/loader.py` loads once at FastAPI startup. The bundle format (`{"model": ..., "features": [...]}`) matters — `loader.py` expects those exact keys, so the API and this notebook never drift out of sync on what columns the model expects, in what order.

`known_coverage_coords` is the set of (rounded) lat/long pairs the model was actually trained on. `has_model_coverage` on the `locations` table is `True` exactly when a resolved location's rounded coordinates appear here — letting the API distinguish "this is a place I have real historical baselines for" from "this is a forecast extrapolated purely from latitude, longitude, and time of year."

**Note:** running this cell will overwrite the artifacts already in `atlas/models/` (which the API is already using). That's intentional for reproducibility — it confirms this notebook produces the same artifacts the live system is built from — but you don't need to re-run it before submission unless you've changed the pipeline.

In [ ]:
forecast_bundle = {"model": model, "features": features}
joblib.dump(forecast_bundle, MODELS_DIR / "forecast_model.joblib")

joblib.dump(baselines, MODELS_DIR / "location_baselines.joblib")

known_coverage_coords = (
    df_model[["latitude", "longitude"]]
    .round(2)
    .drop_duplicates()
    .reset_index(drop=True)
)
joblib.dump(known_coverage_coords, MODELS_DIR / "known_coverage_coords.joblib")

print(f"Saved forecast_model.joblib ({len(features)} features)")
print(f"Saved location_baselines.joblib ({len(baselines)} locations)")
print(f"Saved known_coverage_coords.joblib ({len(known_coverage_coords)} coordinate pairs)")

## 7. Conclusions & Architecture Recap

**What this pipeline produces, and how the live API uses it:**

| Artifact | Built in | Loaded by | Used for |
|---|---|---|---|
| `forecast_model.joblib` | Section 5 | `app/ml/loader.py` at startup | `GET /forecast/{location_id}`, `GET /weather/forecast` — temperature prediction for any lat/long |
| `location_baselines.joblib` | Section 4 | `app/ml/loader.py` at startup | `GET /anomaly-check/{location_id}` — is a live reading unusual for *this specific place* |
| `known_coverage_coords.joblib` | Section 6 | `app/ml/loader.py` at startup | `has_model_coverage` flag on every resolved `Location` row |

**Two cleaning problems, two separate fixes (Section 3):** country-name contamination resolved via coordinate clustering rather than string matching (which would mis-group near-duplicate spellings and miss true duplicates in different scripts/languages); physically impossible readings flagged against hard physical limits rather than statistically, specifically so that genuine extreme weather isn't thrown out alongside sensor errors.

**Two distinct "anomaly" concepts (Sections 3 vs. 4), kept separate on purpose:** a reading can be physically impossible (always an error) or statistically unusual-for-this-location (real signal, surfaced to users) — collapsing these into one z-score check would have hidden genuine extreme-weather events behind the same flag as data errors.

**Model beats the baseline it has to beat (Section 5):** the gradient boosting model is only useful if it outperforms simply looking up "what's normal for this place on this day" — that comparison is run explicitly above, not assumed.

**Limitations / next steps**, if there were more time: the forecast model uses only geography and seasonality (no wind, pressure, or humidity as predictors), so it can't capture short-term weather systems — it's deliberately a climatological baseline forecaster, not a short-horizon meteorological one. A logical next step would be blending this with the OpenWeatherMap forecast API's own short-horizon predictions (already integrated for current conditions) for a hybrid near-term/long-term forecast.